# 6-3 loss.backward()와 .grad 확인 — 기본

직접 작성한 코드와 저장된 실행 결과를 정리했습니다.


In [8]:
import random
import json
import math
import shutil
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, Dataset, random_split

# 실습 결과가 매번 비슷하게 나오도록 seed를 고정합니다.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

device: cpu


In [5]:
w = torch.tensor([1.0], requires_grad=True)
x = torch.tensor([4.0])
y = torch.tensor([10.0])

pred = w * x
# TODO: MSE 형태의 scalar loss를 만드세요.
loss = ((pred - y)**2).mean()

# TODO: backward를 호출하세요.
loss.backward()
print('loss:', loss)
print('w.grad:', w.grad)

loss: tensor(36., grad_fn=<MeanBackward0>)
w.grad: tensor([-48.])


In [6]:
model = nn.Sequential(nn.Linear(4, 5), nn.ReLU(), nn.Linear(5, 3))
x = torch.randn(6, 4)
y = torch.tensor([0, 1, 2, 1, 0, 2])

# TODO: logits와 loss를 계산하세요.
logits = model(x)
loss = nn.CrossEntropyLoss()(logits, y)

# TODO: backward를 호출하세요.
loss.backward()

for name, param in model.named_parameters():
    print(name, 'grad:', None if param.grad is None else tuple(param.grad.shape))

0.weight grad: (5, 4)
0.bias grad: (5,)
2.weight grad: (3, 5)
2.bias grad: (3,)


In [10]:
def grad_norm_summary(model):
    norms = {}
    for name, param in model.named_parameters():
        if param.grad is None:
            continue
        # TODO: gradient norm을 계산하세요.
        norms[name] = param.grad.norm().item()
    avg_norm = sum(norms.values()) / max(len(norms), 1)
    return norms, avg_norm

model = nn.Sequential(nn.Linear(2, 4), nn.Tanh(), nn.Linear(4, 1))
x = torch.randn(8, 2)
y = torch.randn(8, 1)
loss = nn.MSELoss()(model(x), y)
loss.backward()
print(grad_norm_summary(model))

({'0.weight': 0.19015662372112274, '0.bias': 0.31302735209465027, '2.weight': 0.47571027278900146, '2.bias': 1.212766408920288}, 0.5479151643812656)
